In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from m2fgb.m2fgb import M2FGBClassifier

## Loading ACSIncome dataset

In [4]:
from folktables import ACSDataSource, ACSIncome
import os
from copy import deepcopy

data_dir = "data/ACSIncome/"
if not os.path.exists(data_dir):
    os.makedirs(data_dir)

state_list = [
    "AL",
    "AK",
    "AZ",
    "AR",
    "CA",
    "CO",
    "CT",
    "DE",
    "FL",
    "GA",
    "HI",
    "ID",
    "IL",
    "IN",
    "IA",
    "KS",
    "KY",
    "LA",
    "ME",
    "MD",
    "MA",
    "MI",
    "MN",
    "MS",
    "MO",
    "MT",
    "NE",
    "NV",
    "NH",
    "NJ",
    "NM",
    "NY",
    "NC",
    "ND",
    "OH",
    "OK",
    "OR",
    "PA",
    "RI",
    "SC",
    "SD",
    "TN",
    "TX",
    "UT",
    "VT",
    "VA",
    "WA",
    "WV",
    "WI",
    "WY",
]

data_source = ACSDataSource(
    survey_year="2018", horizon="1-Year", survey="person", root_dir=str(data_dir)
)
data = data_source.get_data(states=state_list, download=True)
dataset_details = deepcopy(ACSIncome)
dataset_details.features.append("ST")

features, labels, _ = dataset_details.df_to_numpy(data)
df = pd.DataFrame(data=features, columns=dataset_details.features)
df[dataset_details.target] = labels

# reorder columns
sensitive_col = "SEX"
state_col = "ST"
cols_order = [dataset_details.target, sensitive_col] + list(
    set(dataset_details.features) - {sensitive_col, state_col}
)
df = df[cols_order]

mapping = {
    1: "white",
    2: "african_america",
    3: "american_indian",
    4: "alaska_native",
    5: "american_indian_or_alaska_native",
    6: "asian",
    7: "native_hawaiian",
    8: "other_race",
    9: "two_or_more",
}
df["RAC1P"] = df["RAC1P"].apply(lambda x: mapping[x])

mapping = {1: "male", 2: "female"}
df["SEX"] = df["SEX"].apply(lambda x: mapping[x])

df["PINCP"] = df["PINCP"].apply(
    lambda x: 1 if x is True else 0 if x is False else x
)

# drop columns with many cateogies
df = df.drop(columns=["OCCP", "POBP"])

categorical_columns = ["COW", "SCHL", "MAR", "RELP", "RAC1P", "SEX"]
for col in df.columns:
    if col in categorical_columns:
        df[col] = pd.Categorical(df[col])


data/ACSIncome/2018/1-Year/csv_pal.zip may be corrupted. Please try deleting it and rerunning this command.

Exception:  HTTPSConnectionPool(host='www2.census.gov', port=443): Max retries exceeded with url: /programs-surveys/acs/data/pums/2018/1-Year/csv_pal.zip (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1010)')))

data/ACSIncome/2018/1-Year/csv_pak.zip may be corrupted. Please try deleting it and rerunning this command.

Exception:  HTTPSConnectionPool(host='www2.census.gov', port=443): Max retries exceeded with url: /programs-surveys/acs/data/pums/2018/1-Year/csv_pak.zip (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1010)')))

data/ACSIncome/2018/1-Year/csv_paz.zip may be corrupted. Please try deleting it and rerunning this command.

Exception:  HTTPSConnectionPoo

KeyboardInterrupt: 

In [ ]:
model = M2FGBClassifier(